In [2]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.home() / "Desktop" / "FlowCast"
DATA_PATH = PROJECT_DIR / "data" / "processed" / "flowcast_features.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Dataset shape:", df.shape)
print("\nCongestion labels:")
print(df["congestion_level"].value_counts(dropna=False))

print("\nRequired columns:")
for col in ["road_id", "timestamp", "traffic_volume",
            "road_capacity", "capacity_30min"]:
    print(f"{col}: {'Available' if col in df.columns else 'Missing'}")

Dataset shape: (176701, 46)

Congestion labels:
congestion_level
Free-flow    108544
Moderate      42098
Heavy         16311
Severe         9748
Name: count, dtype: int64

Required columns:
road_id: Available
timestamp: Available
traffic_volume: Available
road_capacity: Available
capacity_30min: Available


In [3]:
print("Missing values in important columns:")
print(
    df[["road_id", "timestamp", "traffic_volume",
        "avg_speed", "congestion_level"]]
    .isnull()
    .sum()
)

print("\nDuplicate road-time records:")
print(df.duplicated(subset=["road_id", "timestamp"]).sum())

print("\nTimestamp range:")
print(df["timestamp"].min(), "to", df["timestamp"].max())

Missing values in important columns:
road_id             0
timestamp           0
traffic_volume      0
avg_speed           0
congestion_level    0
dtype: int64

Duplicate road-time records:
0

Timestamp range:
2025-01-01 00:00:00 to 2025-05-31 23:30:00


In [4]:
print("Traffic volume statistics:")
print(df["traffic_volume"].describe())

print("\nAverage speed statistics:")
print(df["avg_speed"].describe())

print("\nCongestion level percentages:")
print(
    df["congestion_level"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nNumber of roads:", df["road_id"].nunique())

Traffic volume statistics:
count    176701.000000
mean        431.436582
std         283.208213
min          41.000000
25%         195.000000
50%         362.000000
75%         601.000000
max        2090.000000
Name: traffic_volume, dtype: float64

Average speed statistics:
count    176701.000000
mean         42.104921
std          11.375773
min           6.700000
25%          36.200000
50%          44.900000
75%          50.900000
max          64.800000
Name: avg_speed, dtype: float64

Congestion level percentages:
congestion_level
Free-flow    61.43
Moderate     23.82
Heavy         9.23
Severe        5.52
Name: proportion, dtype: float64

Number of roads: 25


In [5]:
import pandas as pd
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score
)

# Sort records chronologically
df = df.sort_values("timestamp").copy()

# Select available prediction features
feature_cols = [
    "traffic_volume",
    "avg_speed",
    "occupancy",
    "hour",
    "minute",
    "day_of_week",
    "month",
    "is_weekend",
    "is_morning_peak",
    "is_evening_peak",
]

feature_cols = [c for c in feature_cols if c in df.columns]

# Keep rows with complete features and target
data = df.dropna(subset=feature_cols + ["congestion_level"]).copy()

# Time-based split: earlier records train, later records test
split = int(len(data) * 0.8)

X_train = data[feature_cols].iloc[:split]
X_test = data[feature_cols].iloc[split:]

y_train = data["congestion_level"].iloc[:split]
y_test = data["congestion_level"].iloc[split:]

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print("Macro F1:", round(f1_score(y_test, predictions, average="macro"), 4))
print("\nClassification report:")
print(classification_report(y_test, predictions, zero_division=0))

Accuracy: 0.956
Macro F1: 0.9394

Classification report:
              precision    recall  f1-score   support

   Free-flow       0.98      0.98      0.98     21300
       Heavy       0.91      0.90      0.91      3497
    Moderate       0.93      0.91      0.92      8524
      Severe       0.95      0.95      0.95      2020

    accuracy                           0.96     35341
   macro avg       0.94      0.94      0.94     35341
weighted avg       0.96      0.96      0.96     35341



In [6]:
from pathlib import Path
import joblib
import json

PROJECT_DIR = Path.home() / "Desktop" / "FlowCast"
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "congestion_classifier.pkl"
features_path = MODEL_DIR / "congestion_features.json"

joblib.dump(model, model_path)

with open(features_path, "w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=4)

print("Congestion model saved:", model_path)
print("Feature list saved:", features_path)

Congestion model saved: C:\Users\pathi\Desktop\FlowCast\models\congestion_classifier.pkl
Feature list saved: C:\Users\pathi\Desktop\FlowCast\models\congestion_features.json


In [7]:
print("Number of congestion features:", len(feature_cols))
print("Congestion features:")
print(feature_cols)

Number of congestion features: 10
Congestion features:
['traffic_volume', 'avg_speed', 'occupancy', 'hour', 'minute', 'day_of_week', 'month', 'is_weekend', 'is_morning_peak', 'is_evening_peak']


In [8]:
# SVM BASELINE MODEL FOR CONGESTION CLASSIFICATION

import os
import joblib
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Sort data chronologically
svm_data = data.sort_values("timestamp").copy()

# Select features and target
X_svm = svm_data[feature_cols]
y_svm = svm_data["congestion_level"]

# Time-based 80/20 split
split_index = int(len(svm_data) * 0.8)

X_train_svm = X_svm.iloc[:split_index]
X_test_svm = X_svm.iloc[split_index:]
y_train_svm = y_svm.iloc[:split_index]
y_test_svm = y_svm.iloc[split_index:]

# Build SVM pipeline
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LinearSVC(
        random_state=42,
        max_iter=5000
    ))
])

# Train model
svm_model.fit(X_train_svm, y_train_svm)

# Predictions
svm_predictions = svm_model.predict(X_test_svm)

# Evaluation
svm_accuracy = accuracy_score(y_test_svm, svm_predictions)
svm_macro_f1 = f1_score(
    y_test_svm, svm_predictions, average="macro"
)

print("SVM BASELINE RESULTS")
print("--------------------")
print("Accuracy:", round(svm_accuracy, 4))
print("Macro F1:", round(svm_macro_f1, 4))
print("\nClassification Report:")
print(classification_report(y_test_svm, svm_predictions))

# Save separately; do not overwrite Random Forest
model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

svm_path = model_dir / "congestion_svm_baseline.pkl"
joblib.dump(svm_model, svm_path)

print("\nSVM model saved:", svm_path)

SVM BASELINE RESULTS
--------------------
Accuracy: 0.8584
Macro F1: 0.6761

Classification Report:
              precision    recall  f1-score   support

   Free-flow       0.94      0.99      0.97     21300
       Heavy       0.91      0.03      0.05      3499
    Moderate       0.67      0.85      0.75      8523
      Severe       0.92      0.95      0.94      2019

    accuracy                           0.86     35341
   macro avg       0.86      0.70      0.68     35341
weighted avg       0.87      0.86      0.82     35341


SVM model saved: ..\models\congestion_svm_baseline.pkl


In [9]:
classification_results = pd.DataFrame([
    {
        "Model": "Congestion Classifier",
        "Macro_F1": 0.9394
    },
    {
        "Model": "SVM Baseline",
        "Macro_F1": 0.6761
    }
])

display(classification_results)

classification_results.to_csv(
    "../reports/classification_model_scoreboard.csv",
    index=False
)

print("Classification scoreboard saved successfully!")

,Model,Macro_F1
0,Congestion Classifier,0.9394
1,SVM Baseline,0.6761


Classification scoreboard saved successfully!
